In [1]:
import pandas as pd

# Load the cleaned and normalized dataset
df = pd.read_csv("lagos_rent_cleaned_locations.csv")

# Verify it loaded correctly
print(f"Data loaded successfully! Total rows: {len(df)}")

# Take a quick peek at the first 5 rows
df.head()

Data loaded successfully! Total rows: 12075


,Location,Price,Is_Serviced,Is_Furnished,Amenities,Raw_Description,URL,Source,Top_Icons,Detailed_Attributes,Cleaned_Location
0,Agungi Lekki Lagos,"₦ 3,800,000/year",0.0,0.0,NaN,? SEMI DIRECT \n \n?FOR RENT: \n ...,https://www.propertypro.ng/property/2-bedroom-...,PropertyPro,NaN,NaN,Agungi Lekki
1,Megamound Ikota Lekki Lagos,"₦ 7,000,000/year",1.0,0.0,NaN,FULLY SERVICE 2 BEDROOM APARTMENT TO LET AT CO...,https://www.propertypro.ng/property/2-bedroom-...,PropertyPro,NaN,NaN,Ikota Lekki Megamound
2,Osapa London Lekki Lagos,"₦ 16,000,000/year",0.0,0.0,NaN,*Well renovated Four 4 bedroom duplex with a B...,https://www.propertypro.ng/property/4-bedroom-...,PropertyPro,NaN,NaN,Lekki London Osapa
3,Ikota Lekki Lagos Ikota Lekki Lagos,"₦ 15,000,000/year",0.0,0.0,NaN,*Newly built four 4 bedroom fully detached dup...,https://www.propertypro.ng/property/4-bedroom-...,PropertyPro,NaN,NaN,Ikota Lekki
4,Ikate Lekki Lagos Ikate Lekki Lagos,"₦ 17,000,000/year",0.0,0.0,NaN,2bedroom apartment with Swimming pool and Gym ...,https://www.propertypro.ng/property/2-bedroom-...,PropertyPro,NaN,NaN,Ikate Lekki


In [1]:
import pandas as pd

# Load your perfectly cleaned and balanced dataset
df = pd.read_csv("lagos_rent_cleaned_locations.csv")

# Print the exact number of properties you have left
print(f"Total clean, mathematically balanced properties ready for ML: {len(df)}")

# (Optional) Take a quick peek to ensure everything looks right
df.head(3)

Total clean, mathematically balanced properties ready for ML: 12075


,Location,Price,Is_Serviced,Is_Furnished,Amenities,Raw_Description,URL,Source,Top_Icons,Detailed_Attributes,Cleaned_Location
0,Agungi Lekki Lagos,"₦ 3,800,000/year",0.0,0.0,NaN,? SEMI DIRECT \n \n?FOR RENT: \n ...,https://www.propertypro.ng/property/2-bedroom-...,PropertyPro,NaN,NaN,Agungi Lekki
1,Megamound Ikota Lekki Lagos,"₦ 7,000,000/year",1.0,0.0,NaN,FULLY SERVICE 2 BEDROOM APARTMENT TO LET AT CO...,https://www.propertypro.ng/property/2-bedroom-...,PropertyPro,NaN,NaN,Ikota Lekki Megamound
2,Osapa London Lekki Lagos,"₦ 16,000,000/year",0.0,0.0,NaN,*Well renovated Four 4 bedroom duplex with a B...,https://www.propertypro.ng/property/4-bedroom-...,PropertyPro,NaN,NaN,Lekki London Osapa


In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("lagos_rent_cleaned_locations.csv")

# Ensure we are looking at the right column
location_col = 'Cleaned_Location' if 'Cleaned_Location' in df.columns else 'Location'

# Calculate the property counts for every single location
location_counts = df[location_col].value_counts().reset_index()
location_counts.columns = ['Neighborhood', 'Property_Count']

# --- THE NEW ALPHABETICAL SORT ---
# Sort the dataframe alphabetically by the Neighborhood name
location_counts = location_counts.sort_values(by='Neighborhood').reset_index(drop=True)

# Print a preview to the terminal
print(f"Total Unique Neighborhoods: {len(location_counts)}\n")
print("--- Alphabetical Preview ---")
print(location_counts.head(20).to_string(index=False))

# Export the FULL list to Excel for your personal inspection
output_file = "Location_Distribution_Check.xlsx"
location_counts.to_excel(output_file, index=False)
print(f"\nSUCCESS: Alphabetized location list saved to '{output_file}'. Open this file to review all neighborhoods!")

Total Unique Neighborhoods: 2574

--- Alphabetical Preview ---
                                                                  Neighborhood  Property_Count
                                        1 12 Johnson Lekki Omorinre Phase Road               1
                                                      1 14 Ago Lekki Min Phase               1
                                                      1 19 Ago Lekki Min Phase               1
                                                    1 1Lekki Lekki Phase State               1
                                               1 1St Cl Lekki Phase Roundabout               1
                                               1 1St Lekki Lt Phase Roundabout               1
                                               1 1St Lekki Md Phase Roundabout               1
                                               1 1St Lekki My Phase Roundabout               1
                                                    1 2 Gra Magodo Ojodu Phase    

In [1]:
import pandas as pd
import re
from thefuzz import process

# 1. Load the uncapped, geofenced dataset
df = pd.read_csv("lagos_rent_cleaned_locations.csv")
target_col = 'Cleaned_Location' if 'Cleaned_Location' in df.columns else 'Location'

# 2. THE TARGETED JANITOR (Preserves order & useful numbers)
def smart_clean_address(text):
    if pd.isna(text): return "Unknown"
    text = str(text).lower()
    
    # Remove commas and periods
    text = re.sub(r'[,.]', ' ', text)
    
    # Target and remove specific address formats (e.g., "plot 4", "block a", "no 12")
    text = re.sub(r'\b(?:plot|block|no|number)\s*\w+\b', '', text)
    
    # Target and remove road types
    text = re.sub(r'\b(?:street|road|avenue|close|crescent|way|ext|extension|highway)\b', '', text)
    
    # Remove the word 'lagos' as it's redundant
    text = re.sub(r'\blagos\b', '', text)
    
    # Clean up extra spaces and convert to Title Case (keeps original word order!)
    text = ' '.join(text.split()).title()
    return text

# Create our first pass draft column
df['Draft_Location'] = df[target_col].apply(smart_clean_address)

# 3. THE FUZZY MATCHER (Round 1: Strict 90% Threshold)
loc_counts = df['Draft_Location'].value_counts()

# The Master List (Locations with 30+ properties)
master_locations = loc_counts[loc_counts >= 30].index.tolist()

# The minor locations we want to check
minor_locations = loc_counts[loc_counts < 30].index.tolist()

fuzzy_map = {}
print("--- ROUND 1: STRICT MERGE PROPOSALS (90%+) ---")

for loc in minor_locations:
    loc_str = str(loc)
    master_strs = [str(m) for m in master_locations]
    
    # STRICT CUTOFF: 90% similarity required
    match = process.extractOne(loc_str, master_strs, score_cutoff=90)
    
    if match:
        best_match_name = match[0]
        similarity_score = match[1]
        fuzzy_map[loc] = best_match_name
        print(f"[{similarity_score}% Match] Merging: '{loc}' ---> '{best_match_name}'")

print(f"\nFound {len(fuzzy_map)} highly confident merges for Round 1.")

--- ROUND 1: STRICT MERGE PROPOSALS (90%+) ---
[90% Match] Merging: '1 Admiralty Lekki Off Phase' ---> 'Lekki'
[90% Match] Merging: 'Efon Igbo Lekki' ---> 'Lekki'
[90% Match] Merging: 'Pedro Shomolu' ---> 'Shomolu'
[90% Match] Merging: 'Yaba' ---> 'Alagomeji Yaba'
[90% Match] Merging: 'Onipan Shomolu' ---> 'Shomolu'
[90% Match] Merging: 'Alternative Chevron Lekki Route' ---> 'Chevron Lekki'
[90% Match] Merging: 'Estate Lekki Orchid Pine Royal' ---> 'Lekki Orchid'
[90% Match] Merging: 'Centre Conservation Lekki' ---> 'Lekki'
[90% Match] Merging: 'Idado Lekki' ---> 'Lekki'
[90% Match] Merging: '1 Gra Magodo Phase' ---> 'Magodo'
[90% Match] Merging: 'Ajah Ogombo' ---> 'Ajah'
[90% Match] Merging: '2 Ajah Lekki Scheme' ---> 'Lekki'
[90% Match] Merging: 'Estate Foreshore Ikoyi Osborne' ---> 'Ikoyi'
[90% Match] Merging: 'Alausa Ikeja' ---> 'Ikeja'
[90% Match] Merging: 'Gbagada' ---> 'Gbagada Ifako'
[90% Match] Merging: 'Ikorodu' ---> 'Ajuwon Akute Ikorodu'
[95% Match] Merging: '1 Ikeja Omole 